In [1]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np
import pandas as pd
import os
import seaborn as sns

from tp7.tp7 import Tape7
# from srfs.srfs import SRFS
import scipy.integrate as integrate
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error, explained_variance_score, mean_absolute_percentage_error
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.neural_network import MLPRegressor
from sklearn.tree import export_text, plot_tree

In [16]:
new_srf_path = "../data/libera_erf_measured_srfs.csv"
old_srf_path = "../data/libera_srfs_20241017.csv"

In [12]:
class SRFS:
    def __init__(self, file_path, old_or_new = 0):
        self.file_path = file_path
        self.old_or_new = old_or_new
        self.srf_vals = None
        self.conversion_df = {}
        self.process_srf()

    def process_srf(self):
        srf = pd.read_csv(self.file_path)
        wv = ""
        if self.old_or_new == 0:
            wv = "wavelength [um]"
        else:
            wv = "wavelength_um"
        # loop through and create a column in the dataframe that holds the end of the wavelength range
        for index, row in srf.iterrows():
            next_index = index + 1
            if next_index not in srf.index:
                continue
            else:
                srf.loc[index, 'End_wv_range'] = srf.loc[next_index, wv]
                # wavelength_um wavelength [um]

        # create the first row in the dataframe starting at wavelenght 0.0 and ending at the first wavelength from the csv
        new_row = srf.iloc[0].copy()
        new_row.loc['End_wv_range'] = new_row[wv]
        new_row.loc[wv] = 0.0
        data, cols = new_row.values, new_row.index
        tdf = pd.DataFrame([data], columns=cols)
        srf = pd.concat([tdf, srf])

        # add the final end value to replace the NAN
        srf.loc[999, 'End_wv_range'] = 1001

        self.srf_vals = srf

        return

In [18]:
new_srf = SRFS(new_srf_path, 1)
new_srf.srf_vals

,wavelength_um,wavelength_um_fwhm,ssw_srf,ssw_srf_sd,lw_srf,lw_srf_sd,total_srf,total_srf_sd,sw_srf,sw_srf_sd,End_wv_range
0,0.0000,0.0012,-0.00206,0.00049,-0.00237,0.00009,0.08595,0.00091,0.06426,0.00146,0.2489
0,0.2489,0.0012,-0.00206,0.00049,-0.00237,0.00009,0.08595,0.00091,0.06426,0.00146,0.2591
1,0.2591,0.0013,0.00012,0.00035,-0.00027,0.00045,0.05688,0.00108,0.04278,0.00083,0.2693
2,0.2693,0.0014,-0.00053,0.00005,-0.00040,0.00046,0.02784,0.00041,0.02145,0.00028,0.2793
3,0.2793,0.0015,-0.00012,0.00000,-0.00018,0.00008,0.01168,0.00010,0.01016,0.00029,0.2895
...,...,...,...,...,...,...,...,...,...,...,...
253,13.4023,1.3140,-0.00051,0.00005,0.68097,0.00152,0.98867,0.00294,-0.00056,0.00060,13.9831
254,13.9831,1.5332,-0.00024,0.00010,0.70013,0.00137,0.98985,0.00202,-0.00117,0.00053,14.7474
255,14.7474,1.8217,-0.00062,0.00026,0.70964,0.00356,0.99105,0.00202,-0.00051,0.00047,15.5442
256,15.5442,2.1224,0.00006,0.00116,0.69462,0.00817,0.99860,0.00752,0.00343,0.00271,NaN


In [17]:
old_srf = SRFS(old_srf_path)
old_srf.srf_vals

,wavelength [um],ssw srf,conv ssw srf,lw srf,conv lw srf,total srf,conv total srf,sw srf,conv sw srf,End_wv_range
0,0.000000,0.0,0.0,0.00000,0.00000,0.00860,0.00865,0.01079,0.01076,0.300000
0,0.300000,0.0,0.0,0.00000,0.00000,0.00860,0.00865,0.01079,0.01076,0.301750
1,0.301750,0.0,0.0,0.00000,0.00000,0.00965,0.00965,0.01006,0.01006,0.303509
2,0.303509,0.0,0.0,0.00000,0.00000,0.01077,0.01076,0.00933,0.00933,0.305279
3,0.305279,0.0,0.0,0.00000,0.00000,0.01182,0.01178,0.00863,0.00865,0.307060
...,...,...,...,...,...,...,...,...,...,...
995,97.700859,0.0,0.0,0.44098,0.45646,0.89716,0.89719,-0.00000,-0.00000,98.270641
996,98.270641,0.0,0.0,0.50463,0.49343,0.89582,0.89590,-0.00000,-0.00000,98.843745
997,98.843745,0.0,0.0,0.56496,0.52561,0.89447,0.89473,-0.00000,-0.00000,99.420192
998,99.420192,0.0,0.0,0.57120,0.54465,0.89312,0.89376,-0.00000,-0.00000,100.000000
